## 2. Load Environment Variables & Initialize Groq LLM
We load your API keys from `.env` (`GROQ_API_KEY`, `TAVILY_API_KEY`). Groq offers ultra-fast inference with native tool calling. Default model is `llama-3.3-70b-versatile` (or `qwen/qwen3.8-27b`, `llama-3.1-8b-instant`).

In [8]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

# Load environment variables
load_dotenv(override=True)

groq_api_key = os.getenv("GROQ_API_KEY") or os.getenv("GROQ_KEY")
tavily_api_key = os.getenv("TAVILY_API_KEY")

if not groq_api_key:
    raise ValueError("GROQ_API_KEY is missing from .env file!")
if not tavily_api_key:
    raise ValueError("TAVILY_API_KEY is missing from .env file!")

# Select Groq model (llama-3.3-70b-versatile provides best-in-class tool calling & synthesis)
GROQ_MODEL = os.getenv("GROQ_MODEL", "qwen/qwen3.8-27b")

llm = ChatGroq(
    model=GROQ_MODEL,
    temperature=0.1,
    api_key=groq_api_key,
    max_retries=2,
)

print(f"✅ Groq LLM Ready: {GROQ_MODEL}")
print(f"✅ Tavily API Key Detected: {'*' * 8 + tavily_api_key[-4:]}")

✅ Groq LLM Ready: qwen/qwen3.8-27b
✅ Tavily API Key Detected: ********2BTH


## 3. Local Filesystem MCP (`@modelcontextprotocol/server-filesystem`)
This integrates the official reference **Filesystem MCP Server** from [Model Context Protocol](https://github.com/modelcontextprotocol/servers/tree/main/src/filesystem).

**Key Advantages:**
- **Local & Instant**: Runs on your machine via `stdio` using `npx -y @modelcontextprotocol/server-filesystem <allowed_dir>`.
- **Zero Authentication**: No OAuth, no Google Cloud Console, no redirect URLs.
- **Full Capabilities**: Grants the agent native tools to write the final market report (`write_file`), read existing notes (`read_file`), list files (`list_directory`), and explore directories.

In [9]:
import os
import sys
import shutil
from langchain_core.tools import tool
from langchain_mcp_adapters.client import MultiServerMCPClient

# Target workspace directory where the agent can read and write reports
WORKSPACE_DIR = os.path.abspath(".")

# Locate npx executable (npx.cmd on Windows, npx on Linux/macOS)
npx_bin = shutil.which("npx") or shutil.which("npx.cmd") or ("npx.cmd" if sys.platform == "win32" else "npx")

fs_mcp_client = None
fs_mcp_tools = []

async def init_filesystem_mcp():
    """Initializes the local Filesystem MCP server via MultiServerMCPClient over stdio."""
    global fs_mcp_client, fs_mcp_tools
    print(f"Connecting to local Filesystem MCP server for: {WORKSPACE_DIR}...")
    try:
        fs_mcp_client = MultiServerMCPClient(
            {
                "filesystem": {
                    "command": npx_bin,
                    "args": [
                        "-y",
                        "@modelcontextprotocol/server-filesystem",
                        WORKSPACE_DIR,
                    ],
                    "transport": "stdio",
                }
            },
            tool_name_prefix=True,
        )
        fs_mcp_tools = await fs_mcp_client.get_tools()
        print(f"✅ Local Filesystem MCP Connected! ({len(fs_mcp_tools)} tools discovered)")
        for t in fs_mcp_tools:
            print(f"   • {t.name}")
        return fs_mcp_tools
    except Exception as e:
        print(f"⚠️ Could not start local Filesystem MCP via npx ({e}).")
        print("   A fallback local file-writer tool will be used.")
        return []

# Initialize Local Filesystem MCP
await init_filesystem_mcp()

# Dedicated helper tool: save_market_report_locally
@tool
async def save_market_report_locally(filename: str, report_content: str) -> str:
    """Writes and saves the final market research report as a local Markdown (.md) file in the workspace.
    
    Args:
        filename: Name of the report file, e.g. 'Market_Research_Report_EV_Egypt.md'.
        report_content: Complete formatted Markdown content of the market research report.
    """
    if not filename.endswith(".md"):
        filename += ".md"
        
    full_path = os.path.join(WORKSPACE_DIR, filename)
    
    # 1. Try via Local Filesystem MCP write_file tool
    write_tool = next((t for t in fs_mcp_tools if "write_file" in t.name.lower()), None)
    if write_tool is not None:
        try:
            res = await write_tool.ainvoke({"path": full_path, "content": report_content})
            return f"✅ Report successfully saved locally via Filesystem MCP!\nFile: {full_path}\nMCP Response: {res}"
        except Exception as err:
            print(f"Filesystem MCP write_file encountered an issue ({err}), falling back to direct write...")
            
    # 2. Direct local file write fallback
    try:
        with open(full_path, "w", encoding="utf-8") as f:
            f.write(report_content)
        return f"✅ Report successfully written locally!\n• Path: {full_path}\n• Size: {len(report_content)} characters"
    except Exception as e:
        return f"❌ Failed to write report locally: {e}"

print("✅ Local report writer tool ready!")

Connecting to local Filesystem MCP server for: c:\Users\kirok\Desktop\New folder (6)...
⚠️ Could not start local Filesystem MCP via npx (fileno).
   A fallback local file-writer tool will be used.
✅ Local report writer tool ready!


## 4. Google Drive MCP Integration & Cloud Export (Optional)
If you wish to export reports to the cloud (Google Drive), this section:
1. Detects `client_secret_*.json` or `cre.json`.
2. Connects to the **Google Drive MCP server** (`https://drivemcp.googleapis.com/mcp/v1`) using `MultiServerMCPClient`.
3. If Drive MCP is not enabled on your Google Cloud project, it gracefully falls back to the Google Drive v3 REST API using the same OAuth credentials.

In [10]:
import glob
import json
import time
import requests
import urllib.parse
from langchain_mcp_adapters.client import MultiServerMCPClient

def find_google_credentials_file():
    preferred = "client_secret_548126423526-bq8hds2erpmhd6t244j95m39t04qrvv8.apps.googleusercontent.com.json"
    if os.path.exists(preferred):
        return preferred
    client_secrets = glob.glob("client_secret_*.json")
    if client_secrets:
        return client_secrets[0]
    for fallback in ["cre.json", "credentials.json"]:
        if os.path.exists(fallback):
            return fallback
    return None

def parse_google_credentials(filepath=None):
    if filepath is None:
        filepath = find_google_credentials_file()
    if not filepath or not os.path.exists(filepath):
        return None, None, [], None
    with open(filepath, "r", encoding="utf-8") as f:
        content = f.read().strip()
    try:
        data = json.loads(content)
        info = data.get("web") or data.get("installed") or data
        cid = info.get("client_id")
        sec = info.get("client_secret")
        redirect_uris = info.get("redirect_uris", [])
        if cid and sec:
            return cid.strip(), sec.strip(), redirect_uris, filepath
    except Exception:
        pass
    return None, None, [], filepath

def get_google_drive_access_token(token_file="token.json"):
    """Checks token_file or environment for an existing active Google Drive token."""
    env_token = os.getenv("GOOGLE_DRIVE_MCP_TOKEN")
    if env_token:
        return env_token
    if os.path.exists(token_file):
        try:
            with open(token_file, "r", encoding="utf-8") as f:
                tdata = json.load(f)
            expires_at = tdata.get("expires_at", 0)
            if time.time() < (expires_at - 60) and tdata.get("access_token"):
                return tdata["access_token"]
            refresh_token = tdata.get("refresh_token")
            cid, sec, _, _ = parse_google_credentials()
            if refresh_token and cid and sec:
                resp = requests.post(
                    "https://oauth2.googleapis.com/token",
                    data={
                        "client_id": cid,
                        "client_secret": sec,
                        "refresh_token": refresh_token,
                        "grant_type": "refresh_token"
                    },
                    timeout=20
                )
                if resp.status_code == 200:
                    new_data = resp.json()
                    tdata["access_token"] = new_data["access_token"]
                    tdata["expires_at"] = time.time() + new_data.get("expires_in", 3600)
                    with open(token_file, "w", encoding="utf-8") as f:
                        json.dump(tdata, f, indent=2)
                    return tdata["access_token"]
        except Exception:
            pass
    return None

drive_token = get_google_drive_access_token()
mcp_drive_client = None
mcp_drive_create_tool = None

async def init_drive_mcp():
    global mcp_drive_client, mcp_drive_create_tool
    if not drive_token:
        print("ℹ️ Google Drive token not currently configured (continuing with Local Filesystem MCP).")
        return None
    try:
        mcp_drive_client = MultiServerMCPClient(
            {
                "google_drive": {
                    "url": "https://drivemcp.googleapis.com/mcp/v1",
                    "transport": "streamable_http",
                    "headers": {"Authorization": f"Bearer {drive_token}"},
                }
            },
            tool_name_prefix=True,
        )
        tools = await mcp_drive_client.get_tools()
        create_tools = [t for t in tools if "create_file" in t.name.lower()]
        if create_tools:
            mcp_drive_create_tool = create_tools[0]
            print(f"✅ Google Drive MCP Connected: {mcp_drive_create_tool.name}")
            return mcp_drive_create_tool
    except Exception as e:
        print(f"ℹ️ Google Drive MCP notice: {e} (Direct Drive REST upload fallback ready).")
    return None

await init_drive_mcp()

def _direct_drive_api_upload(token: str, filename: str, content: str):
    url = "https://www.googleapis.com/upload/drive/v3/files?uploadType=multipart"
    headers = {"Authorization": f"Bearer {token}"}
    metadata = {"name": filename, "mimeType": "text/markdown"}
    files = {
        "data": ("metadata", json.dumps(metadata), "application/json; charset=UTF-8"),
        "file": (filename, content.encode("utf-8"), "text/markdown"),
    }
    res = requests.post(url, headers=headers, files=files, timeout=30)
    res.raise_for_status()
    return res.json()

@tool
async def write_market_report_to_drive(filename: str, report_content: str) -> str:
    """Writes and saves the final market research report to Google Drive as a Markdown file.
    
    Args:
        filename: Name of the report file, e.g. 'Market_Research_Report.md'.
        report_content: Complete formatted Markdown content of the market research report.
    """
    if not filename.endswith(".md"):
        filename += ".md"
    token = drive_token or get_google_drive_access_token()
    if not token:
        return "❌ Error: Google Drive token is not configured. Use save_market_report_locally to save to local disk instead."
        
    if mcp_drive_create_tool is not None:
        try:
            res = await mcp_drive_create_tool.ainvoke({"name": filename, "content": report_content, "mimeType": "text/markdown"})
            return f"✅ Saved to Google Drive via Drive MCP! File: {filename}\nResult: {res}"
        except Exception as err:
            print(f"Drive MCP upload failed ({err}); falling back to Drive REST API...")
            
    try:
        file_info = _direct_drive_api_upload(token, filename, report_content)
        file_id = file_info.get("id")
        drive_url = f"https://drive.google.com/file/d/{file_id}/view"
        return f"✅ Successfully uploaded report to Google Drive!\n• File Name: {filename}\n• File ID: {file_id}\n• Google Drive View Link: {drive_url}"
    except Exception as e:
        return f"❌ Failed to upload report to Google Drive: {type(e).__name__}: {e}"

print("✅ Cloud Drive report writer tool ready!")

ℹ️ Google Drive MCP notice: unhandled errors in a TaskGroup (1 sub-exception) (Direct Drive REST upload fallback ready).
✅ Cloud Drive report writer tool ready!


## 5. Market Research Tools (Powered by Tavily Search API)
The agent uses specialized Tavily search tools tailored for market intelligence:
1. `tavily_market_overview`: Market size, CAGR growth projections, macro trends, industry drivers, and regulatory environment.
2. `tavily_competitor_analysis`: Direct and indirect competitors, market share breakdown, product features, value propositions, and SWOT analysis.
3. `tavily_pricing_and_demand`: Retail pricing tiers (budget/mid/premium), discounts, top e-commerce platforms, customer demand signals, and purchase drivers.

In [11]:
from langchain_tavily import TavilySearch

# Core Tavily Search engine
tavily_search_engine = TavilySearch(
    max_results=5,
    topic="general",
    search_depth="advanced"
)

@tool
def tavily_market_overview(query: str, country_or_region: str = "") -> str:
    """Researches overall market size, growth trajectory, CAGR, industry drivers, and market dynamics.
    
    Args:
        query: Market or industry topic, e.g. 'electric vehicles', 'skincare', 'fintech'.
        country_or_region: Target geography, e.g. 'Egypt', 'UAE', 'Saudi Arabia', 'Global'.
    """
    search_q = f"{query} market size growth trends demand outlook {country_or_region}".strip()
    try:
        results = tavily_search_engine.invoke({"query": search_q})
        return results if isinstance(results, str) else json.dumps(results, ensure_ascii=False, indent=2)
    except Exception as e:
        return f"Market overview search failed: {e}"

@tool
def tavily_competitor_analysis(market_or_product: str, competitors_or_brands: str = "") -> str:
    """Researches key competitor brands, market shares, positioning, feature matrices, and competitive advantages.
    
    Args:
        market_or_product: Product category or market sector.
        competitors_or_brands: Specific brand names, competitors, or 'top players'.
    """
    search_q = f"{market_or_product} top competitors market share brands {competitors_or_brands} comparison analysis".strip()
    try:
        results = tavily_search_engine.invoke({"query": search_q})
        return results if isinstance(results, str) else json.dumps(results, ensure_ascii=False, indent=2)
    except Exception as e:
        return f"Competitor analysis search failed: {e}"

@tool
def tavily_pricing_and_demand(product_or_category: str, target_market: str = "") -> str:
    """Researches retail price points, pricing tiers, discounts, retailer platforms, and customer demand signals.
    
    Args:
        product_or_category: Product or service category.
        target_market: Target country or geographic market.
    """
    search_q = f"{product_or_category} price range prices retailers sellers customer demand signals {target_market}".strip()
    try:
        results = tavily_search_engine.invoke({"query": search_q})
        return results if isinstance(results, str) else json.dumps(results, ensure_ascii=False, indent=2)
    except Exception as e:
        return f"Pricing and demand search failed: {e}"

# Combine all tools for the agent
research_tools = [
    tavily_market_overview,
    tavily_competitor_analysis,
    tavily_pricing_and_demand,
]

export_tools = [
    save_market_report_locally,
    write_market_report_to_drive,
]

all_agent_tools = research_tools + export_tools + fs_mcp_tools

print(f"🛠️ Total Registered Tools: {len(all_agent_tools)}")
for t in all_agent_tools:
    print(f"  • {t.name}: {t.description.splitlines()[0]}")

🛠️ Total Registered Tools: 5
  • tavily_market_overview: Researches overall market size, growth trajectory, CAGR, industry drivers, and market dynamics.
  • tavily_competitor_analysis: Researches key competitor brands, market shares, positioning, feature matrices, and competitive advantages.
  • tavily_pricing_and_demand: Researches retail price points, pricing tiers, discounts, retailer platforms, and customer demand signals.
  • save_market_report_locally: Writes and saves the final market research report as a local Markdown (.md) file in the workspace.
  • write_market_report_to_drive: Writes and saves the final market research report to Google Drive as a Markdown file.


## 6. Build the LangGraph Market Researcher Agent
We construct a **LangGraph ReAct agent** with:
- **`agent` node**: LLM with tool-calling capabilities and a detailed Senior Market Analyst system prompt.
- **`tools` node**: LangGraph `ToolNode` executing Tavily searches, Local Filesystem MCP tools, and Drive uploads.
- **`tools_condition`**: Automatically routes between executing tools and returning the synthesized report.
- Full multi-turn conversation memory and citation verification.

In [12]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition

SYSTEM_PROMPT = """You are an elite Senior Market Research Analyst and Competitive Intelligence Strategist.
Your mission is to conduct thorough, empirical market research on requested products, industries, and geographic regions using your Tavily research tools, and synthesize high-impact reports.

### Operational Guidelines:
1. Grounded & Factual: Base findings strictly on data retrieved via your search tools. Never hallucinate market sizes, revenues, or statistics. Always cite source URLs.
2. Multi-Dimensional Research:
   - Investigate Market Sizing, Growth (CAGR), and Drivers using `tavily_market_overview`.
   - Benchmark Competitors, Market Shares, and Product Matrices using `tavily_competitor_analysis`.
   - Analyze Pricing Tiers, Retailers, and Consumer Demand Signals using `tavily_pricing_and_demand`.
3. Report Saving & Exporting:
   - To save the report locally: call `save_market_report_locally` (or `filesystem__write_file` from Local Filesystem MCP) with a clean filename (e.g., 'Market_Research_Report_<Topic>_<Region>.md') and the complete markdown content.
   - To save to Google Drive (if user explicitly asks for Drive): call `write_market_report_to_drive`.
4. Report Output Format:
   Your final report must be formatted in professional GitHub-Flavored Markdown:
   - # [Title] Comprehensive Market & Competitor Research Report
   - ## 1. Executive Summary & Key Takeaways
   - ## 2. Market Overview & Industry Dynamics (Size, Growth, Trends)
   - ## 3. Competitor Landscape & Benchmarking Matrix (Key Players, Market Share, Value Props)
   - ## 4. Pricing Tiers, Retail Landscape & Consumer Demand Signals
   - ## 5. Market Opportunities, Gaps & Strategic Risks (SWOT-style breakdown)
   - ## 6. Strategic Recommendations & Actionable Next Steps
   - ## 7. Verified Sources & Citations (Direct URLs from research)
5. Execution Protocol:
   - Execute necessary research tool calls first.
   - If local saving was requested, call `save_market_report_locally`.
   - Present the full, beautifully rendered report in your final text response.
""".strip()

class MarketAgentState(MessagesState):
    pass

llm_with_tools = llm.bind_tools(all_agent_tools)

def agent_node(state: MarketAgentState):
    messages = state["messages"]
    if not any(isinstance(m, SystemMessage) for m in messages):
        messages = [SystemMessage(content=SYSTEM_PROMPT)] + messages
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

tool_node = ToolNode(all_agent_tools)

builder = StateGraph(MarketAgentState)
builder.add_node("agent", agent_node)
builder.add_node("tools", tool_node)

builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition, {"tools": "tools", END: END})
builder.add_edge("tools", "agent")

market_graph = builder.compile()
print("✅ Market Researcher LangGraph Agent compiled successfully!")

✅ Market Researcher LangGraph Agent compiled successfully!


## 7. Execution Runner & Display Helper
The helper function below streams agent events, prints tool invocations in real-time, and displays the final Markdown report in notebook output.

In [13]:
from IPython.display import display, Markdown

async def run_market_research(user_prompt: str, verbose: bool = True):
    """Runs the Market Researcher Agent asynchronously and renders the final report."""
    print("\n" + "=" * 70)
    print("🚀 LAUNCHING MARKET RESEARCH AGENT")
    print(f"📋 Prompt: {user_prompt}")
    print("=" * 70 + "\n")
    
    state_input = {"messages": [HumanMessage(content=user_prompt)]}
    final_report = None
    
    async for event in market_graph.astream(state_input, config={"recursion_limit": 25}):
        for node_name, node_state in event.items():
            messages = node_state.get("messages", [])
            for msg in messages:
                if verbose:
                    if isinstance(msg, AIMessage) and msg.tool_calls:
                        for tc in msg.tool_calls:
                            print(f"🔧 [Tool Call] -> {tc['name']}")
                            print(f"   Args: {json.dumps(tc['args'], ensure_ascii=False)}")
                    elif isinstance(msg, ToolMessage):
                        preview = str(msg.content)[:180].replace("\n", " ") + "..."
                        print(f"📥 [Tool Result] <- {msg.name}: {preview}")
                if isinstance(msg, AIMessage) and not msg.tool_calls:
                    final_report = msg.content

    print("\n" + "=" * 70)
    print("🏁 MARKET RESEARCH COMPLETE!")
    print("=" * 70 + "\n")
    
    if final_report:
        display(Markdown(final_report))
    return final_report

## 8. Test 1: Market Research + Save via Local Filesystem MCP
This test conducts market and competitor research and saves the complete final report directly to your local workspace using the **Local Filesystem MCP tool** (`@modelcontextprotocol/server-filesystem`).

In [14]:
prompt_1 = (
    "Conduct an in-depth market research and competitor analysis on the Electric Vehicle (EV) Charging Infrastructure market in Egypt. "
    "Identify key competitors and network operators, pricing models, consumer adoption barriers, and emerging market opportunities. "
    "Generate a complete, professional Market Research Report and save it as a local Markdown file using the Local Filesystem MCP as 'EV_Charging_Egypt_Market_Report.md'."
)

report_1 = await run_market_research(prompt_1, verbose=True)


🚀 LAUNCHING MARKET RESEARCH AGENT
📋 Prompt: Conduct an in-depth market research and competitor analysis on the Electric Vehicle (EV) Charging Infrastructure market in Egypt. Identify key competitors and network operators, pricing models, consumer adoption barriers, and emerging market opportunities. Generate a complete, professional Market Research Report and save it as a local Markdown file using the Local Filesystem MCP as 'EV_Charging_Egypt_Market_Report.md'.

🔧 [Tool Call] -> tavily_competitor_analysis
   Args: {"competitors_or_brands": "top players, EV charging network operators", "market_or_product": "EV charging infrastructure"}
📥 [Tool Result] <- tavily_competitor_analysis: {   "query": "EV charging infrastructure top competitors market share brands top players, EV charging network operators comparison analysis",   "follow_up_questions": null,   "answ...
🔧 [Tool Call] -> tavily_market_overview
   Args: {"country_or_region": "Egypt", "query": "EV charging infrastructure market"

APIStatusError: Error code: 413 - {'error': {'message': 'Request too large for model `qwen/qwen3.8-27b` in organization `org_01kyyv40b6fknrk4w03w032f0r` service tier `on_demand` on input tokens per minute (ITPM): Limit 7000, Requested 10341, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

## 9. Test 2: Market Research + Google Drive Cloud Upload
This scenario tests exporting directly to Google Drive via the Drive MCP / API (requires active Google token).

In [15]:
prompt_2 = (
    "Research the Smart Wearable / Fitness Tracker market in Saudi Arabia and the UAE. "
    "Compare the top competitor brands (Apple, Samsung, Garmin, Xiaomi, etc.), retail pricing tiers, and consumer demand signals. "
    "Generate a structured market report with verified citations, and upload it to my Google Drive as 'Smart_Wearables_Gulf_Market_Report.md'."
)

report_2 = await run_market_research(prompt_2, verbose=True)


🚀 LAUNCHING MARKET RESEARCH AGENT
📋 Prompt: Research the Smart Wearable / Fitness Tracker market in Saudi Arabia and the UAE. Compare the top competitor brands (Apple, Samsung, Garmin, Xiaomi, etc.), retail pricing tiers, and consumer demand signals. Generate a structured market report with verified citations, and upload it to my Google Drive as 'Smart_Wearables_Gulf_Market_Report.md'.

🔧 [Tool Call] -> tavily_market_overview
   Args: {"country_or_region": "Saudi Arabia and UAE", "query": "smart wearable fitness tracker market size growth CAGR"}
🔧 [Tool Call] -> tavily_competitor_analysis
   Args: {"competitors_or_brands": "Apple, Samsung, Garmin, Xiaomi, Fitbit, Huawei", "market_or_product": "smart wearables and fitness trackers"}
🔧 [Tool Call] -> tavily_pricing_and_demand
   Args: {"product_or_category": "smartwatches and fitness trackers", "target_market": "Saudi Arabia and UAE"}
📥 [Tool Result] <- tavily_market_overview: {   "query": "smart wearable fitness tracker market size gro

APIStatusError: Error code: 413 - {'error': {'message': 'Request too large for model `qwen/qwen3.8-27b` in organization `org_01kyyv40b6fknrk4w03w032f0r` service tier `on_demand` on input tokens per minute (ITPM): Limit 7000, Requested 8821, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}